# Reproducible LSTM Training Experiment

This notebook demonstrates a short CPU-friendly Phase 8 experiment. It trains on the training split, uses validation loss for early stopping, and evaluates the selected state afterward. It does not backtest or claim profitability.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.comparison import compare_classification_models, compare_regression_models
from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.neural.alignment import create_dated_sequences
from ml.neural.config import NeuralConfig
from ml.neural.dataset import FinancialSequenceDataset
from ml.neural.loaders import create_sequence_loader
from ml.neural.lstm import LSTMClassifier, LSTMRegressor
from ml.supervised import build_supervised_dataset
from ml.training.config import TrainingConfig
from ml.training.evaluation import evaluate_lstm
from ml.training.history import TrainingHistory
from ml.training.trainer import train_lstm
from ml.visualization.training_plots import plot_training_history

## Load AAPL and Build Supervised Data

In [ ]:
raw_path = project_root / 'data' / 'raw' / 'AAPL.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    ohlcv = MarketDataIngestionService(YahooFinanceProvider()).ingest('AAPL', '2020-01-01', '2026-01-01')
regression_data = build_supervised_dataset(ohlcv, target_type='regression', horizon=1)
classification_data = build_supervised_dataset(ohlcv, target_type='direction', horizon=1)

## Create Independent Sequences and DataLoaders

In [ ]:
lookback = 20
def make_loaders(data):
    sequence_values = [create_dated_sequences(X.to_numpy(), y.to_numpy(), dates, lookback) for X, y, dates in ((data.X_train, data.y_train, data.dates_train), (data.X_validation, data.y_validation, data.dates_validation))]
    datasets = [FinancialSequenceDataset(*values) for values in sequence_values]
    return [create_sequence_loader(dataset, 32) for dataset in datasets]
regression_train_loader, regression_validation_loader = make_loaders(regression_data)
classification_train_loader, classification_validation_loader = make_loaders(classification_data)

## Train Regression LSTM

In [ ]:
model_config = NeuralConfig(lookback=lookback, input_size=len(regression_data.feature_names), hidden_size=32, batch_size=32, seed=42)
training_config = TrainingConfig(epochs=10, patience=3, device='cpu', seed=42)
regression_model = LSTMRegressor(model_config.input_size, model_config.hidden_size)
regression_result = train_lstm(regression_model, regression_train_loader, regression_validation_loader, task='regression', configuration=training_config, metadata={'ticker': 'AAPL', 'horizon': 1, 'lookback': lookback})
print('best epoch:', regression_result.best_epoch)
figure, axes = plot_training_history(regression_result.history)
figure.show()

## Evaluate Regression and Compare Baselines

In [ ]:
lstm_regression_validation = evaluate_lstm(regression_result.model, regression_validation_loader, task='regression')
regression_baselines = compare_regression_models(regression_data)
print(lstm_regression_validation)
print(pd.DataFrame([{'model': result.model_name, **result.metrics} for result in regression_baselines]))

## Train and Evaluate Direction LSTM

In [ ]:
classification_model = LSTMClassifier(model_config.input_size, model_config.hidden_size)
classification_result = train_lstm(classification_model, classification_train_loader, classification_validation_loader, task='classification', configuration=training_config, metadata={'ticker': 'AAPL', 'horizon': 1, 'lookback': lookback})
classification_validation = evaluate_lstm(classification_result.model, classification_validation_loader, task='classification')
classification_baselines = compare_classification_models(classification_data)
print('best epoch:', classification_result.best_epoch)
print(classification_validation)
print(pd.DataFrame([{'model': result.model_name, **{key: value for key, value in result.metrics.items() if key != 'confusion_matrix'}} for result in classification_baselines]))

## Limitations

This is a small reproducible training demonstration, not evidence of market predictability or profitability. Test evaluation should occur only after model selection. Walk-forward evaluation, checkpoint policy integration, and experiment tracking remain future work. No trading strategy or backtest is included.